In [ ]:
# Construir pipeline ETL con manejo de errores completo

In [ ]:
# Registro de configuración:

In [9]:
import logging
import time
from functools import wraps

# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('etl_ecommerce.log'),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger('etl_ecommerce')

def log_etapa(etapa):
    """Decorator para logging de etapas"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            logger.info(f"🚀 Iniciando {etapa}")
            start_time = time.time()
            
            try:
                result = func(*args, **kwargs)
                duration = time.time() - start_time
                logger.info(f"✅ {etapa} completada en {duration:.2f}s")
                return result
            except Exception as e:
                duration = time.time() - start_time
                logger.error(f"💥 {etapa} falló en {duration:.2f}s: {e}")
                raise e
        
        return wrapper
    return decorator

In [ ]:
# Gestión de errores de ETL en pipeline:

In [10]:
import pandas as pd
import numpy as np
from typing import Dict, Any

class ETLPipeline:
    def __init__(self):
        self.logger = logger
        self.errores = []
    
    @log_etapa("extracción de datos")
    def extract(self) -> pd.DataFrame:
        """Extraer datos con manejo de errores"""
        try:
            # Simular extracción (podría fallar)
            if np.random.random() < 0.1:  # 10% chance de error
                raise ConnectionError("Error de conexión a fuente de datos")
            
            # Datos de ejemplo
            datos = pd.DataFrame({
                'orden_id': range(1, 101),
                'cliente_id': np.random.randint(1, 21, 100),
                'producto': np.random.choice(['A', 'B', 'C', 'D'], 100),
                'cantidad': np.random.randint(1, 6, 100),
                'precio': np.round(np.random.uniform(10, 200, 100), 2)
            })
            
            self.logger.info(f"Extraídos {len(datos)} registros")
            return datos
            
        except Exception as e:
            self.errores.append(f"Extract: {e}")
            raise e
    
    @log_etapa("transformación de datos")
    def transform(self, datos: pd.DataFrame) -> pd.DataFrame:
        """Transformar datos con validaciones"""
        try:
            df = datos.copy()
            
            # Validar datos de entrada
            if df.empty:
                raise ValueError("No hay datos para transformar")
            
            # Transformaciones
            df['total'] = df['cantidad'] * df['precio']
            df['categoria_precio'] = pd.cut(
                df['precio'], 
                bins=[0, 50, 100, 200], 
                labels=['Bajo', 'Medio', 'Alto']
            )
            
            # Validar transformaciones
            if df['total'].isnull().any():
                raise ValueError("Transformación produjo valores nulos")
            
            self.logger.info(f"Transformados {len(df)} registros")
            return df
            
        except Exception as e:
            self.errores.append(f"Transform: {e}")
            raise e
    
    @log_etapa("carga de datos")
    def load(self, datos: pd.DataFrame) -> bool:
        """Cargar datos con verificación"""
        try:
            # Simular carga (podría fallar)
            if np.random.random() < 0.05:  # 5% chance de error
                raise Exception("Error de conexión a base de datos")
            
            # En producción: datos.to_sql('ventas', engine, if_exists='append')
            self.logger.info(f"Cargados {len(datos)} registros exitosamente")
            
            # Validar carga
            registros_esperados = len(datos)
            registros_cargados = len(datos)  # Simulado
            
            if registros_cargados != registros_esperados:
                raise ValueError(f"Carga incompleta: {registros_cargados}/{registros_esperados}")
            
            return True
            
        except Exception as e:
            self.errores.append(f"Load: {e}")
            raise e
    
    def ejecutar_pipeline(self) -> Dict[str, Any]:
        """Ejecutar pipeline completo con manejo de errores"""
        self.logger.info("🎯 Iniciando pipeline ETL completo")
        
        try:
            # Extract
            datos_crudo = self.extract()
            
            # Transform
            datos_transformados = self.transform(datos_crudo)
            
            # Load
            exito = self.load(datos_transformados)
            
            resultado = {
                'exito': True,
                'registros_procesados': len(datos_transformados),
                'errores': self.errores
            }
            
            self.logger.info("🎉 Pipeline ETL completado exitosamente")
            return resultado
            
        except Exception as e:
            self.logger.error(f"🚨 Pipeline ETL falló: {e}")
            
            return {
                'exito': False,
                'error_principal': str(e),
                'errores': self.errores
            }

In [ ]:
# Ejecutar y validar pipeline:

In [11]:
# Ejecutar pipeline con diferentes escenarios
pipeline = ETLPipeline()

# Ejecución exitosa
resultado = pipeline.ejecutar_pipeline()

print("\nResultado del pipeline:")
print(f"Éxito: {resultado['exito']}")
if resultado['exito']:
    print(f"Registros procesados: {resultado['registros_procesados']}")
else:
    print(f"Error principal: {resultado['error_principal']}")

print(f"Errores registrados: {len(resultado['errores'])}")
for error in resultado['errores']:
    print(f"  - {error}")

# Ejecutar múltiples veces para probar robustez
resultados_multiples = []
for i in range(5):
    print(f"\n--- Ejecución {i+1} ---")
    pipeline_i = ETLPipeline()
    resultado_i = pipeline_i.ejecutar_pipeline()
    resultados_multiples.append(resultado_i['exito'])

exito_rate = sum(resultados_multiples) / len(resultados_multiples)
print(f"Tasa de éxito: {exito_rate * 100:.1f}%")

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\mfram\anaconda31\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\mfram\anaconda31\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\U0001f3af' in position 33: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\mfram\anaconda31\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\mfram\anaconda31\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "C:\Users\mfram\anaconda31\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.st

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\mfram\anaconda31\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\mfram\anaconda31\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\U0001f680' in position 33: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\mfram\anaconda31\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\mfram\anaconda31\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "C:\Users\mfram\anaconda31\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.st

2026-01-15 19:08:54,806 - INFO - Cargados 100 registros exitosamente
--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\mfram\anaconda31\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\mfram\anaconda31\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2705' in position 33: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\mfram\anaconda31\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\mfram\anaconda31\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "C:\Users\mfram\anaconda31\Lib\site-package

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\mfram\anaconda31\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\mfram\anaconda31\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\U0001f680' in position 33: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\mfram\anaconda31\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\mfram\anaconda31\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "C:\Users\mfram\anaconda31\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.st

2026-01-15 19:08:54,877 - INFO - Transformados 100 registros
--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\mfram\anaconda31\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\mfram\anaconda31\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2705' in position 33: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\mfram\anaconda31\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\mfram\anaconda31\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "C:\Users\mfram\anaconda31\Lib\site-packages\ipyker

...

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\mfram\AppData\Local\Temp\ipykernel_22268\1977948000.py", line 26, in wrapper
    result = func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\mfram\AppData\Local\Temp\ipykernel_22268\1177871329.py", line 32, in extract
    raise e
  File "C:\Users\mfram\AppData\Local\Temp\ipykernel_22268\1177871329.py", line 16, in extract
    raise ConnectionError("Error de conexión a fuente de datos")
ConnectionError: Error de conexión a fuente de datos

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\mfram\anaconda31\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\mfram\anaconda31\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: '

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\mfram\anaconda31\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\mfram\anaconda31\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\U0001f3af' in position 33: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\mfram\anaconda31\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\mfram\anaconda31\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "C:\Users\mfram\anaconda31\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.st

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\mfram\anaconda31\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\mfram\anaconda31\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\U0001f680' in position 33: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\mfram\anaconda31\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\mfram\anaconda31\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "C:\Users\mfram\anaconda31\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.st

  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\mfram\anaconda31\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\mfram\anaconda31\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "C:\Users\mfram\anaconda31\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\mfram\anaconda31\Lib\site-packages\tornado\platform\asyncio.py", line 195, in start
    self.asyncio_loop.run_forever()
  File "C:\Users\mfram\anaconda31\Lib\asyncio\windows_events.py", line 321, in run_forever
    super().run_forever()
  File "C:\Users\mfram\anaconda31\Lib\asyncio\base_events.py", line 607, in run_forever
    self._run_once()
  File "C:\Users\mfram\anaconda31\Lib\asyncio\base_events.py", line 1922, in _run_once
    handle._run()
  File "C:\Users\mfram\anaconda31\Lib\asyncio\ev


--- Ejecución 4 ---

--- Ejecución 5 ---
Tasa de éxito: 80.0%


In [ ]:
""" 
¿Qué información debería incluir en los logs para facilitar el debugging? 

Lo ideal es registrar cuatro capas de información:

1. Contexto de ejecución 
    • Timestamp de inicio y fin de cada etapa
    • Nombre del pipeline y versión
    • Parámetros de entrada
    • Entorno (dev / qa / prod)
    • ID de correlación o  único
    • Usuario o proceso que disparó la ejecución
    Esto permite reproducir el escenario exacto.

2. Logs por etapa del pipeline
    Cada fase del ETL debería loguear:
    • Inicio y fin de la etapa
    • Cantidad de registros leídos / transformados / cargados
    • Tiempo de ejecución
    • Fuente y destino involucrados
    • Validaciones aplicadas
    Esto te permite saber dónde se degradó el rendimiento o dónde se rompió la lógica.

3. Errores y excepciones (lo más importante)
    Incluye:
    • Tipo de error (ValueError, ConnectionError, etc.)
    • Mensaje completo de la excepción
    • Stack trace
    • Registro o lote que causó el error
    • Query SQL o fragmento de código relevante
    • Datos sensibles anonimizados (si aplica)
    La clave es que el log permita responder:
    “¿Qué dato exacto provocó el problema?”

4. Métricas operacionales
    • Registros procesados por etapa
    • Registros descartados
    • Registros corregidos automáticamente
    • Retries realizados
    • Uso de memoria o tiempo total del pipeline
    Esto alimenta dashboards de calidad y confiabilidad.

¿Cómo decides entre continuar el pipeline con errores parciales vs detenerlo completamente?

Detener el pipeline inmediatamente cuando:
    • El error afecta la integridad global del dataset
    • La fuente está caída o inaccesible
    • La transformación produce datos inconsistentes
    • La carga puede generar duplicados, corrupción o sobreescritura incorrecta
    • No se puede garantizar la calidad mínima del resultado
    Ejemplos típicos:
    • No se puede conectar a la base OLTP
    • El esquema cambió y las columnas ya no coinciden
    • La tabla destino quedó en estado inconsistente
    Aquí lo correcto es fallar rápido, registrar todo y evitar daños mayores.

Continuar con errores parciales cuando:
    • El error afecta solo a un subconjunto de registros
    • El pipeline está diseñado para tolerancia a fallas
    • Puedes aislar, corregir o descartar registros problemáticos
    • El negocio prefiere “procesar lo que se pueda”
    • Hay mecanismos de reintento o colas de errores
    Ejemplos:
    • 5 de 10.000 registros vienen con fecha inválida
    • Un archivo CSV tiene 2 filas corruptas
    • Una API devolvió error en un subset de IDs
    Aquí lo correcto es:
    • Procesar lo sano
    • Registrar lo dañado
    • Guardar los registros fallidos en una tabla de errores
    • Notificar al equipo
"""